In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;

In [0]:
from pyspark.sql.functions import col, trim

# ================================
# 1. READ BRONZE TABLE
# ================================

df = spark.table("electronics_retailer_clg.bronze.stores")


# ================================
# 2. CLEAN COLUMN NAMES
# ================================

df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])


# ================================
# 3. TRIM SPACES
# ================================

for c in df.columns:
    df = df.withColumn(c, trim(col(c)))


# ================================
# 4. FIX DATA TYPES
# ================================

df = df.withColumn("storekey", col("storekey").cast("int"))


# ================================
# 5. HANDLE NULLS
# ================================

df = df.fillna({
    "country": "unknown"
})


# ================================
# 6. KEEP ONLY REQUIRED COLUMNS ✅
# ================================

df = df.select(
    "storekey",
    "country"
)


# ================================
# 7. REMOVE DUPLICATES
# ================================

df = df.dropDuplicates(["storekey"])


# ================================
# 8. FINAL CHECK
# ================================

display(df)
df.printSchema()


# ================================
# 9. WRITE TO SILVER
# ================================

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("electronics_retailer_clg.silver.stores")

print("✅ Store cleaned successfully")